# Day 17｜U-Net

> **来源**：U-Net 原论文（Ronneberger et al., 2015）｜待补具体课程/视频链接
> **目标**：能手写出在 28×28 上跑通、输入输出同尺寸的 U-Net
> **前置**：day13 卷积 ｜ day14 尺寸公式 ｜ day16 残差与 BN
> **复习状态**：首学 09-13 ｜ 已复习 0 次 ｜ 自测未做

---

## 0. 五秒回忆卡（闭卷先答）

1. U-Net 为什么必须是对称的 U 形，只做下采样行不行？
2. skip 是 `cat` 还是 `add`？和 ResNet 的差别在哪？
3. 28×28 经过两层 2×2 MaxPool 后，为什么还能回到 28×28？

---

## 1. 一句话定义

图像分割网络：输入一张图，输出**同尺寸**的逐像素标签图；同时也是扩散模型（DDPM）的去噪骨干。

和分类网络的区别：分类把整图压成一个标签、丢掉位置；分割必须同时知道"是什么"（语义）和"在哪"（位置）。

---

## 2. 结构：三段 + 一条捷径

| 部位 | 做什么 | 尺寸变化 |
| --- | --- | --- |
| 编码器（下采样） | 抓语义、扩大感受野、省计算 | H、W ÷2，通道 ×2 |
| 瓶颈 | U 字底部，最抽象的语义 | 尺寸最小、通道最多 |
| 解码器（上采样） | 把语义铺回每个像素、恢复位置 | H、W ×2 |
| skip 拼接 | 把编码器同尺寸层的细节（边缘、位置）补给解码器 | 通道相加 |

28×28 的干净路径：`28 → 14 → 7`（瓶颈）`→ 14 → 28`

---

## 3. 形状规律（写代码时逐层对照）

- 下采样 1 层：H、W ÷2，通道 ×2
- 上采样 1 层：H、W ×2；`F.interpolate` 不改通道数，`ConvTranspose2d` 通常会减半
- 拼接：通道直接相加（64 + 128 = 192），再用卷积压回
- 输出层：1×1 卷积把通道变成类别数，空间尺寸保持不变
- 验收：`unet(torch.randn(1, 1, 28, 28)).shape == (1, 1, 28, 28)`

---

## 4. 代码骨架

- `DoubleConv`：conv → BN → ReLU ×2（3×3、padding=1，只改通道不改尺寸）
- 下采样：`MaxPool2d(2)` 之后接 `DoubleConv`
- 上采样：`F.interpolate(scale_factor=2, mode="bilinear")`（或可学习的 `ConvTranspose2d`）
- 拼接：`torch.cat([skip, x], dim=1)`，`dim=1` 是通道维（B, C, H, W）
- 输出：`Conv2d(32, 1, kernel_size=1)`

forward 顺序（默写用）：

```python
e0 = double_conv(x)                       # (1,  32, 28, 28)
e1 = down(e0)                             # (1,  64, 14, 14)
e2 = down(e1)                             # (1, 128,  7,  7)   瓶颈
x  = up(e2) -> cat(e1) -> double_conv     # (1,  64, 14, 14)
x  = up(x)  -> cat(e0) -> double_conv     # (1,  32, 28, 28)
out = conv1x1(x)                          # (1,   1, 28, 28)
```

---

## 5. 与 ResNet 捷径的区别（必考对比）

| | ResNet | U-Net |
| --- | --- | --- |
| 融合方式 | 逐元素相加 `Y + X` | 沿通道拼接 `cat` |
| 通道要求 | 必须一致，不一致用 1×1 卷积对齐 | 允许不一致，通道直接叠加 |
| 融合之后 | 直接过激活 | 一般再接一次卷积把通道压回 |

---

## 6. 易错点

- `cat` 的 `dim` 写成 0 会拼到 batch 维，直接报错或形状错乱。
- 输入尺寸要能一路整除到瓶颈：`28 → 14 → 7` 可以；`30 → 15 → 7`（MaxPool 向下取整）`→ 14 → 28`，回不到 30。
- 去掉 skip 的 U-Net 就是普通 autoencoder：形状照样跑通，但分割边缘会糊。

---

## 7. 和 DDPM 的关系（这次学它的原因）

- 同一套结构当作去噪骨干：输入是含噪图 + 时间步 t，输出是**预测的噪声**。
- 相对分割版要改三处：通道从 1 变成 3（RGB）、BN 换成 GroupNorm（batch 太小）、每个尺度注入时间步嵌入。
- 在 16×16 那一层插入 self-attention——QKV 会在这里登场。

---

## 8. 自测 3 题

1. 默写 forward 顺序，并标出每一步的形状。
2. 解释 skip 为什么用 `cat` 而不是 `add`，各自要求什么前提。
3. 把输入换成 32×32，下采样次数要怎么改？写出完整尺寸链。

---

## 9. 待办

- [ ] 补一个可直接运行的 U-Net 脚本（仓库里还没有，笔记里原先提到的 `outputs/unet_simple.py` 目前不存在）
- [ ] 跑通一个 smoke test：随机张量进、同尺寸张量出
